In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from data.fetcher import OptionChainFetcher
from data.cleaner import clean_option_chain
from models.surface import IVSurface
from analytics.geometry import (
    compute_skew,
    compute_curvature,
    compute_25delta_skew,
    compute_wing_curvature,
    compute_all_geometry_metrics
)

In [ ]:
# Fetch and build surface
fetcher = OptionChainFetcher('SPY')
chain = fetcher.fetch_current_chain()
clean_chain = clean_option_chain(chain)
surface = IVSurface(clean_chain)

In [ ]:
# Compute all geometry metrics
metrics = compute_all_geometry_metrics(surface)

for key, value in metrics.items():
    if isinstance(value, (int, float)):
        print(f"{key}: {value:.6f}")

In [ ]:
# Skew across expiries
expiries = [14, 21, 30, 45, 60, 90]
skews = []

for exp in expiries:
    smile = surface.get_smile(exp)
    if smile is not None:
        skew = compute_skew(smile)
        skews.append({'expiry': exp, 'skew': skew})

skew_df = pd.DataFrame(skews)

fig = go.Figure()
fig.add_trace(go.Bar(
    x=skew_df['expiry'],
    y=skew_df['skew'],
    marker_color='indianred'
))
fig.update_layout(
    title='Skew by Expiry',
    xaxis_title='Days to Expiry',
    yaxis_title='Skew (dIV/dK)'
)
fig.show()

In [ ]:
# Curvature visualization
curvatures = []

for exp in expiries:
    smile = surface.get_smile(exp)
    if smile is not None:
        curv = compute_curvature(smile)
        curvatures.append({'expiry': exp, 'curvature': curv})

curv_df = pd.DataFrame(curvatures)

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=curv_df['expiry'],
    y=curv_df['curvature'],
    mode='lines+markers',
    marker=dict(size=10),
    line=dict(width=2)
))
fig.update_layout(
    title='Curvature by Expiry',
    xaxis_title='Days to Expiry',
    yaxis_title='Curvature (d2IV/dK2)'
)
fig.show()

In [ ]:
# Wing curvature analysis
left_wing, right_wing = compute_wing_curvature(surface, 30)

print(f"Left wing curvature (puts): {left_wing:.6f}")
print(f"Right wing curvature (calls): {right_wing:.6f}")
print(f"\nWing asymmetry: {abs(left_wing - right_wing):.6f}")